**PSO: Particle Swarm Optimization**

A mathematical modelling algorithm, where the goal is to find the global minima of a fitness function.

**PSO with Classification Models**

To ensure a fair comparison against Encoders with genetic algorithm, the PSO will be trained on NBC model, then run by SVM and RF models.

In [5]:
import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

X_df = pd.read_excel('minmax.xlsx')
y_df = pd.read_csv('idC_with_header.csv')

X = X_df.to_numpy()
y = y_df.to_numpy()

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=X.shape[1], options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = X[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / X.shape[1]
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_selected = X[:, selected]

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)
print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_train, y_train)
nbc_preds = nbc_model.predict(X_test)
nbc_acc = accuracy_score(y_test, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy on features selected by PSO: {rf_acc * 100:.4f}")

2025-04-19 17:12:31,732 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.0917
2025-04-19 17:20:06,958 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.09168464419475662, best pos: [0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1
 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0
 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1
 0 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1
 1 0 0 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0
 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0
 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 1
 1 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 0 1 0 1
 1 1 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1 1 

Best Cost:  0.09168464419475662 
Best POS:  [0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 1 1 0 0 1
 0 0 0 0 1 1 0 1 0 0 1 1 0 0 0 0 1 1 0 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 1 1 0
 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 1
 0 1 1 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 0 1
 1 0 0 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0
 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 1 0 1 0 0 0 0 1 0 1 0 1 0 1 1 0 1 0
 1 1 0 1 0 0 0 0 1 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 1
 1 1 1 1 1 0 1 1 1 0 0 1 0 0 1 0 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 0 1 0 1
 1 1 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 0 0 0 1 1 0
 1 0 0 0 0 0 0 1 0 1 0 0 1 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 0 0 1 1 1 0 1 1 1
 0 1 0 0 0 1 1 0 1 0 0 1 0 1 1 1 0 0 0 1 0 0 1 1 0 1 1 0 1 1 0 0 0 0 1 0 1
 0 0 1 0 1 0 1 1 0 0 1 0 1 1 0 1 0 0 0 0 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 0 0
 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 0 0 0 1 0 0 0 1 1 0 1 0 0

**Summary**

Accuracy results for models is slightly less than that of Encoders with GA implementation.

- SVM with PSO: 95%
- NBC with PSO: 87%
- RF with PSO: 93%

**AE + PSO**

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=latent_dim, options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = train_latent[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y_train_np,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / latent_dim
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_trainselected = train_latent[:, selected]
X_testselected = test_latent[:, selected]

print("Selected features: ", selected)

svm_model = SVC(random_state=42)
svm_model.fit(X_trainselected, y_train_np)
svm_preds = svm_model.predict(X_testselected)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_trainselected, y_train_np)
nbc_preds = nbc_model.predict(X_testselected)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_trainselected, y_train_np)
rf_preds = rf_model.predict(X_testselected)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on features selected by PSO: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1633
Epoch [2/50], Loss: 0.8256
Epoch [3/50], Loss: 0.6668
Epoch [4/50], Loss: 0.5592
Epoch [5/50], Loss: 0.4584
Epoch [6/50], Loss: 0.4516
Epoch [7/50], Loss: 0.4993
Epoch [8/50], Loss: 0.4608
Epoch [9/50], Loss: 0.4646
Epoch [10/50], Loss: 0.4069
Epoch [11/50], Loss: 0.4700
Epoch [12/50], Loss: 0.3963
Epoch [13/50], Loss: 0.3647
Epoch [14/50], Loss: 0.3707
Epoch [15/50], Loss: 0.3788
Epoch [16/50], Loss: 0.4003
Epoch [17/50], Loss: 0.4012
Epoch [18/50], Loss: 0.3569
Epoch [19/50], Loss: 0.4100
Epoch [20/50], Loss: 0.4024
Epoch [21/50], Loss: 0.3683
Epoch [22/50], Loss: 0.3937
Epoch [23/50], Loss: 0.4194
Epoch [24/50], Loss: 0.4352
Epoch [25/50], Loss: 0.3448
Epoch [26/50], Loss: 0.3673
Epoch [27/50], Loss: 0.3834
Epoch [28/50], Loss: 0.4055
Epoch [29/50], Loss: 0.3934
Epoch [30/50], Loss: 0.4357
Epoch [31/50], Loss: 0.3851
Epoch [32/50], Loss: 0.3894
Epoch [33/50], Loss: 0.3618
Epoch [34/50], Loss: 0.3844
Epoch [35/50], Loss: 0.4030


2025-04-25 09:11:41,919 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}


Epoch [50/50], Loss: 0.3901


pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.00125
2025-04-25 09:12:39,485 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.0012500000000000011, best pos: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 1]


Best Cost:  0.0012500000000000011 
Best POS:  [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 1]
Selected features:  [ 0 18 23 31]
SVM Accuracy on features selected by PSO: 91.0112
Naïve Bayes Accuracy on features selected by PSO: 93.2584
Random Forest Accuracy on features selected by PSO: 94.3820


**VAE + PSO**

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()


import pyswarms as ps
from pyswarms.discrete import BinaryPSO
from pyswarms.utils.search import RandomSearch
from pyswarms.single import GlobalBestPSO
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

options = {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}
pso = BinaryPSO(n_particles=30, dimensions=latent_dim, options=options)

import numpy as np
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def fitness(particles):
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)
    alpha = 0.99
    for i, particle in enumerate(particles):
        mask = particle.astype(bool)
        if not mask.any():
            scores[i] = 1.0
            continue
        X_sub = train_latent[:, mask]
        acc = cross_val_score(GaussianNB(var_smoothing=1e-9),
                              X_sub, y_train_np,
                              cv=5,
                              scoring='accuracy',
                              n_jobs=-1).mean()
        sparsity = mask.sum() / latent_dim
        scores[i] = alpha * (1 - acc) + (1 - alpha) * sparsity
    return scores

best_cost, best_pos = pso.optimize(fitness, iters=100)
print("Best Cost: ", best_cost, "\nBest POS: ", best_pos)

selected = np.where(best_pos == 1)[0]
X_trainselected = train_latent[:, selected]
X_testselected = test_latent[:, selected]

print("Selected features: ", selected)

svm_model = SVC(random_state=42)
svm_model.fit(X_trainselected, y_train_np)
svm_preds = svm_model.predict(X_testselected)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on features selected by PSO: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(X_trainselected, y_train_np)
nbc_preds = nbc_model.predict(X_testselected)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on features selected by PSO: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_trainselected, y_train_np)
rf_preds = rf_model.predict(X_testselected)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on features selected by PSO: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.4654
Epoch [2/50], Loss: 1.2654
Epoch [3/50], Loss: 1.1564
Epoch [4/50], Loss: 1.0982
Epoch [5/50], Loss: 0.9904
Epoch [6/50], Loss: 0.9175
Epoch [7/50], Loss: 0.9095
Epoch [8/50], Loss: 0.8446
Epoch [9/50], Loss: 0.8084
Epoch [10/50], Loss: 0.7910
Epoch [11/50], Loss: 0.7647
Epoch [12/50], Loss: 0.7563
Epoch [13/50], Loss: 0.7398
Epoch [14/50], Loss: 0.7532
Epoch [15/50], Loss: 0.6826
Epoch [16/50], Loss: 0.7888
Epoch [17/50], Loss: 0.7466
Epoch [18/50], Loss: 0.7589
Epoch [19/50], Loss: 0.7351
Epoch [20/50], Loss: 0.7292
Epoch [21/50], Loss: 0.7269
Epoch [22/50], Loss: 0.7529
Epoch [23/50], Loss: 0.6705
Epoch [24/50], Loss: 0.7129
Epoch [25/50], Loss: 0.6824
Epoch [26/50], Loss: 0.7262
Epoch [27/50], Loss: 0.7165
Epoch [28/50], Loss: 0.6528
Epoch [29/50], Loss: 0.6727
Epoch [30/50], Loss: 0.6320
Epoch [31/50], Loss: 0.7465
Epoch [32/50], Loss: 0.6583
Epoch [33/50], Loss: 0.6926
Epoch [34/50], Loss: 0.7129
Epoch [35/50], Loss: 0.6673


2025-04-25 09:16:59,463 - pyswarms.discrete.binary - INFO - Optimize for 100 iters with {'c1': 2, 'c2': 2, 'w': 0.9, 'k': 5, 'p': 2}


Epoch [50/50], Loss: 0.6645


pyswarms.discrete.binary: 100%|██████████|100/100, best_cost=0.00125
2025-04-25 09:18:21,643 - pyswarms.discrete.binary - INFO - Optimization finished | best cost: 0.0012500000000000011, best pos: [0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]


Best Cost:  0.0012500000000000011 
Best POS:  [0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
Selected features:  [ 1  3  4 24]
SVM Accuracy on features selected by PSO: 88.7640
Naïve Bayes Accuracy on features selected by PSO: 88.7640
Random Forest Accuracy on features selected by PSO: 89.8876
